⏱️ **Time required:** ~2 minutes | **Type:** Cross-Domain Mesh Pipeline (run all cells)

# 🕸️ RideFlow Data Mesh — Cross-Domain Products

**Domain Owner:** Platform / Data Mesh Team

This notebook represents the culmination of the Data Mesh architecture. Instead of running its own Bronze → Silver pipelines against raw data, this pipeline **only consumes published Data Products from other domains**.

It demonstrates:
1. **Data Product Discovery:** Identifying available data products from all domain lakehouses.
2. **Cross-Domain Joins:** Creating a unified `Marketplace × Operations × Payments` view.
3. **Mesh-Level KPIs:** Aggregating platform-wide metrics.

---
## Step 1 · Environment Setup

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import lakelogic as ll

PROJECT_ROOT = Path(".").resolve()
LAKEHOUSE = PROJECT_ROOT / "lakehouse"

print(f"Project Root : {PROJECT_ROOT}")
print(f"Lakehouse    : {LAKEHOUSE}")

### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

---
## Step 2 · Data Product Discovery

A consumer team (or the Data Mesh platform itself) can scan the Data Catalog to discover what data products are currently published and available across the organization.

In [ ]:
print("🔍 Scanning Organization for Data Products...\n")

# In a real ecosystem, this queries the Unity Catalog or Glue.
# Here we simulate it by scanning the lakehouse directories.

domains = ["marketplace", "payments", "operations", "marketing"]
products_found = []

for domain in domains:
    silver_layer = LAKEHOUSE / domain / "silver"
    gold_layer = LAKEHOUSE / domain / "gold"

    for layer in [silver_layer, gold_layer]:
        if layer.exists():
            for path in layer.iterdir():
                if path.is_dir() and (path / "_delta_log").exists():
                    products_found.append(
                        {"domain": domain, "layer": layer.name, "product_name": path.name, "path": str(path)}
                    )

df_catalog = pl.DataFrame(products_found)

print(f"Found {len(df_catalog)} active data products across {len(domains)} domains.")
display(df_catalog.sort(["domain", "layer"]))

---
## Step 3 · Cross-Domain Aggregation: The Unified Rider View

The Customer 360 team wants to combine data to score the overall Rider Experience.

**Join Required:**
1. `gold_rideflow_rider_lifetime_value` (Marketplace)
2. `silver_zendesk_support_tickets` (Operations)
3. `gold_stripe_payment_reconciliation` (Payments)

In [ ]:
# Load the Mesh Products
rider_ltv_path = LAKEHOUSE / "marketplace" / "gold" / "gold_rideflow_rider_lifetime_value"
tickets_path = LAKEHOUSE / "operations" / "silver" / "silver_zendesk_support_tickets"
payments_path = LAKEHOUSE / "payments" / "gold" / "gold_stripe_payment_reconciliation"

ready = True
for p in [rider_ltv_path, tickets_path, payments_path]:
    if not p.exists():
        print(f"⚠️ Missing required data product: {p.name}")
        ready = False

if ready:
    rider_ltv = pl.read_delta(str(rider_ltv_path))
    tickets = pl.read_delta(str(tickets_path))
    payments = pl.read_delta(str(payments_path))

    # 1. Summarize Support Tickets per Rider (Operations Domain)
    # We extract rider_id from the ticket description for correlation
    rider_tickets = tickets.with_columns(
        pl.col("ticket_description").str.extract(r"(RDR-\d+)", 1).alias("rider_id")
    ).filter(pl.col("rider_id").is_not_null())

    ticket_summary = rider_tickets.group_by("rider_id").agg(
        [
            pl.count("ticket_id").alias("support_ticket_count"),
            pl.col("priority").filter(pl.col("priority").is_in(["high", "urgent"])).count().alias("escalated_tickets"),
        ]
    )

    # 2. Summarize Failed Payments per Rider (Payments Domain)
    failed_payments = (
        payments.filter(pl.col("status") == "failed")
        .group_by("rider_id")
        .agg(pl.count("charge_id").alias("failed_payments_count"))
    )

    # 3. Create the Unified View
    unified_view = (
        rider_ltv.join(ticket_summary, on="rider_id", how="left")
        .join(failed_payments, on="rider_id", how="left")
        .fill_null(0)
    )

    # Health Score Metric: Combines LTV with operational friction
    unified_view = unified_view.with_columns(
        (
            (pl.col("total_spend") / 10) - (pl.col("support_ticket_count") * 5) - (pl.col("failed_payments_count") * 10)
        ).alias("experience_score")
    ).sort("total_spend", descending=True)

    print("🌟 Unified Cross-Domain View: Customer 360")
    display(unified_view.head(15))

else:
    print(
        "\n❌ Cannot build cross-domain view. Please ensure the Marketplace, Operations, and Payments notebooks have completed their runs."
    )

---
## 🏆 Conclusion

You have successfully orchestrated a Data Mesh!

1. **Domain Autonomy:** Data generators and pipelines lived inside distinct, team-owned notebooks.
2. **Data Contracts:** Governance (SLOs, Schema, Privacy) was strictly enforced via YAML registries.
3. **Data Products:** High-quality, trusted datasets were discovered and consumed universally.
4. **Mesh Interoperability:** Teams combined independently managed data to create complex architectural value without monolithic bottlenecks.